# SparseChron: Complete Kaggle Setup (Crash-Resilient)

This notebook is fully automated and resumable. Click **Run All**!

## How checkpoint persistence actually works on Kaggle
`/kaggle/working` is wiped when an interactive session dies, so checkpoints only survive across sessions if you **chain notebook versions**:

1. Run the notebook. Time-based checkpoints land in `/kaggle/working/outputs`.
2. When the session ends (or you stop it), click **Save Version** so `/kaggle/working` is captured as the version output.
3. In the next session, click **+ Input → Your Work → (this notebook, latest version)** to attach the previous output, then **Run All** again.
4. The *Restore checkpoints* cell below automatically copies any `.ckpt` files from the attached input into `/kaggle/working/outputs`, and training resumes from the latest one.

**Requirements:** GPU accelerator (T4) + Internet on (phone-verified account) for git/pip/wget. The sanity cell below fails fast if either is missing.

In [ ]:
import subprocess, shutil, torch

out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout[:800] or 'nvidia-smi not found!')
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Enable it: Settings -> Accelerator -> GPU T4 x2, then re-run.'
assert shutil.which('git'), 'git missing - enable Internet in Settings, then re-run.'

In [ ]:
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y git ninja-build libglib2.0-0 libsm6 libxrender-dev libxext6 ffmpeg > /dev/null 2>&1
print('apt packages installed')

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('SparseChron'):
    !git clone https://github.com/rajhodedara/SparseChron.git
else:
    os.chdir('SparseChron')
    !git stash
    !git pull --rebase
    os.chdir('..')

In [ ]:
import os, subprocess, torch

os.chdir('/kaggle/working/SparseChron')

# 1) Try a prebuilt gsplat wheel matched to this torch/CUDA combo.
#    This skips the 10-20 minute JIT compile that otherwise burns session time.
pt_tag = 'pt' + '.'.join(torch.__version__.split('+')[0].split('.')[:2])   # e.g. pt25
cuda_tag = 'cu' + (torch.version.cuda or '').replace('.', '')[:3]          # e.g. cu121
wheel_index = f'https://docs.gsplat.studio/whl/{pt_tag}{cuda_tag}'
print(f'Trying prebuilt gsplat wheel from {wheel_index} ...')
r = subprocess.run(['pip', 'install', '--quiet', '--index-url', wheel_index, 'gsplat>=1.0.0'])
if r.returncode != 0:
    print('Prebuilt wheel unavailable - falling back to source build (slower).')
    env = {**os.environ, 'MAX_JOBS': '4'}  # cap compile RAM on Kaggle
    subprocess.run(['pip', 'install', '--quiet', 'gsplat>=1.0.0'], env=env, check=True)

# 2) Everything else from requirements.txt (already-installed packages are skipped)
subprocess.run(['pip', 'install', '--quiet', '-r', 'requirements.txt'], check=True)

import gsplat
print('gsplat', gsplat.__version__, 'ready.')

In [ ]:
import os
os.chdir('/kaggle/working')
# Tip: attach the HyperNeRF zip as a Kaggle Dataset input to skip this ~1.6GB download.
if not os.path.exists('interp_cut-lemon.zip'):
    !wget -q https://github.com/google/hypernerf/releases/download/v0.1/interp_cut-lemon.zip
if not os.path.exists('cut-lemon1'):
    !unzip -q interp_cut-lemon.zip
    !mv interp_cut-lemon cut-lemon1
print('Dataset ready.')

In [ ]:
import os, shutil, glob

os.makedirs('/kaggle/working/outputs', exist_ok=True)
restored = 0
# Works when a previous version's output is attached via "+ Input -> Your Work".
for src in glob.glob('/kaggle/input/**/outputs/*.ckpt', recursive=True):
    dst = f'/kaggle/working/outputs/{os.path.basename(src)}'
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        restored += 1
print(f'Restored {restored} checkpoint(s) from attached inputs.')
print(sorted(os.listdir('/kaggle/working/outputs')) or 'No checkpoints yet - fresh start.')

In [ ]:
%cd /kaggle/working/SparseChron
import os
if not os.path.exists('/kaggle/working/data/cameras.json'):
    !PYTHONPATH=/kaggle/working/SparseChron python scripts/convert_hypernerf.py --scene-dir /kaggle/working/cut-lemon1 --output-dir /kaggle/working/data
else:
    print('Dataset already converted. Skipping...')

In [ ]:
%cd /kaggle/working/SparseChron
# Outputs go to /kaggle/working/outputs (persisted when you Save Version).
# Training auto-resumes from the latest checkpoint restored in the cell above.
# Val renders + PSNR land in /kaggle/working/outputs/val every 2000 iters.
!PYTHONPATH=/kaggle/working/SparseChron python scripts/train.py --scene-dir /kaggle/working/data --output-dir /kaggle/working/outputs --is-4d --mixed-precision --resume-from latest --seed 42

In [ ]:
import os, glob, subprocess

os.chdir('/kaggle/working/SparseChron')
ckpts = sorted(glob.glob('/kaggle/working/outputs/*.ckpt'))
if ckpts:
    env = {**os.environ, 'PYTHONPATH': '/kaggle/working/SparseChron'}
    subprocess.run([
        'python', 'scripts/evaluate.py',
        '--checkpoint-path', ckpts[-1],
        '--dataset-path', '/kaggle/working/data',
        '--output-dir', '/kaggle/working/eval',
    ], env=env)
else:
    print('No checkpoints found - run training first.')

## After your run

1. **Save Version** so `/kaggle/working` is captured as this version's output.
2. **Next session:** attach this notebook's latest version output as an Input (**+ Input → Your Work**), then Run All — the restore cell copies the checkpoints into `/kaggle/working/outputs` and training continues from there.
3. Check `outputs/val/*.png` for periodic pred-vs-GT previews and the `[val]` PSNR logs — this is how you spot black renders or floaters mid-session on Kaggle.
4. Final metrics land in `/kaggle/working/eval/metrics.json` (PSNR / SSIM / LPIPS).